<a href="https://colab.research.google.com/github/max-santos-d/estudo-persistencia-colab/blob/main/base_persist%C3%AAncia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Configuração e persistência do SQLite no Google Colab

## 1. Arquitetura de persistência proposta


> Google Colab → Ambiente temporário de execução → Intância descartável \
> ↓↑ Recuperação e sincronização \
> SQLite local — `/content/` → Cópia de trabalho da instância atual → Temporário \
> ↓↑ Backup consistente \
> Google Drive → Arquivo SQLite persistente → Persistência

Por que utilizar essa arquitetura?

Existem duas possibilidades principais:

1. SQLite diretamente no Google Drive: o banco é aberto e manipulado diretamente no diretório montado do Drive.
2. SQLite local com sincronização para o Google Drive: o banco é manipulado em `/content/` e sincronizado com o Drive por meio de backups.

Será utilizada a segunda abordagem.

O SQLite utiliza mecanismos de bloqueio de arquivos e pode trabalhar com arquivos auxiliares, como os arquivos WAL. O Google Drive montado no Colab não oferece as mesmas características de um sistema de arquivos local convencional. Por isso, trabalhar diretamente sobre o banco armazenado no Drive pode ocasionar problemas de desempenho, bloqueio e sincronização.

Na segunda abordagem:

- O banco é carregado do Drive quando uma nova instância começa.
- As operações SQL são realizadas localmente.
- O banco é sincronizado com o Drive quando solicitada a persistência.
- Quando uma instância é destruída, o arquivo persistente permanece no Drive.
- Uma nova instância recupera o último backup realizado.

**Importante**: a sincronização não é automática. Se a instância for encerrada antes do backup, as alterações posteriores à última sincronização serão perdidas. O processo será estruturado de forma a facilitar essa operação.

## 2. Organização dos projetos no Google Drive

Antes de inicializar o SQLite, será estabelecida uma estrutura de diretórios que permita trabalhar com vários bancos de dados independentes.

Recomenda-se criar a seguinte estrutura:

```
Meu Drive/Colab_Notebooks
│
└── SQLite_Projetos/
    │
    ├── projeto_01/
    │   └── banco.sqlite3
    │
    ├── projeto_02/
    │   └── banco.sqlite3
    │
    ├── projeto_03/
    │   └── banco.sqlite3
    │
    └── projeto_04/
        └── banco.sqlite3
```

Cada projeto terá seu próprio diretório e seu próprio arquivo SQLite.

| Elemento | Responsabilidade |
| :---: | :---: |
| `SQLite_Projetos/` | Diretório central dos bancos SQLite |
| `projeto_01/` | Diretório exclusivo de um projeto |
| `banco.sqlite3` | Arquivo físico do banco de dados |

**Regra de organização**: cada projeto deverá possuir um identificador exclusivo, que será utilizado pelo notebook para localizar e recuperar seu respectivo banco de dados.

Assim, não será necessário modificar toda a configuração do SQLite quando iniciar um novo projeto. Bastará alterar o identificador.

## 3. Inicialização do Google Drive no Colab

A primeira etapa consiste em conectar o Google Drive ao ambiente de execução do Colab.

Crie uma célula de código no notebook:

In [2]:
# Célula 1 — Montar Google Drive

from google.colab import drive

drive.mount('/content/drive/')

Mounted at /content/drive/


Execute a célula e autorize o acesso à sua conta Google.

Após a execução, o Drive estará acessível pelo caminho: `/content/drive/MyDrive/Colab_Notebooks/SQLite_Projetos`

In [3]:
# Célula 2 — Criar estrutura de armazenamento

from pathlib import Path

# Diretório central no Google Drive
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/Colab_Notebooks/SQLite_Projetos"
)

# Criar diretório caso ainda não exista
DRIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Diretório de armazenamento: {DRIVE_ROOT}")
print(f"Diretório existente: {DRIVE_ROOT.exists()}")

Diretório de armazenamento: /content/drive/MyDrive/Colab_Notebooks/SQLite_Projetos
Diretório existente: True


**Essa operação é idempotente**: executá-la novamente não criará diretórios duplicados nem apagará os arquivos existentes.

## 4. Definição e identificação do projeto

Definição de qual banco de dados será utilizado naquela execução.

In [4]:
# Célula 3 — Identificar projeto

# Identificador exclusivo do projeto
PROJECT_ID = "_base-persistencia"

# Diretório persistente do projeto
PROJECT_DRIVE_DIR = DRIVE_ROOT / PROJECT_ID

# Diretório de trabalho temporário no Colab
PROJECT_LOCAL_DIR = Path(
    "/content/sqlite_workspace"
) / PROJECT_ID

# Arquivos físicos do banco
DRIVE_DB = PROJECT_DRIVE_DIR / "banco.sqlite3"

LOCAL_DB = PROJECT_LOCAL_DIR / "banco.sqlite3"

# Criar diretórios necessários
PROJECT_DRIVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PROJECT_LOCAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Projeto: {PROJECT_ID}")
print(f"Banco persistente: {DRIVE_DB}")
print(f"Banco de trabalho: {LOCAL_DB}")

Projeto: _base-persistencia
Banco persistente: /content/drive/MyDrive/Colab_Notebooks/SQLite_Projetos/_base-persistencia/banco.sqlite3
Banco de trabalho: /content/sqlite_workspace/_base-persistencia/banco.sqlite3


Observe a diferença entre os dois caminhos:
- DRIVE_DB: localização permanente do banco no Google Drive.
- LOCAL_DB: localização temporária do banco na máquina virtual do Colab.

Quando quiser trabalhar em outro projeto, basta modificar: `PROJECT_ID = "projeto_02"`

O notebook passará a trabalhar com outro banco, sem misturar os dados dos projetos anteriores.

**Atenção**: não altere o identificador durante uma sessão sem antes persistir o banco atual. Cada identificador representa um banco independente.

## 5. Implementação da recuperação e persistência do SQLite

Esta é a etapa central.

Vamos criar duas operações fundamentais:

1. **Recuperação**: carregar o banco persistente do Google Drive para a instância atual do Colab.
2. **Persistência**: gerar uma cópia consistente do banco local e armazená-la no Google Drive.

Para realizar essas operações, utilizaremos a funcionalidade `Connection.backup()` disponibilizada pelo módulo nativo `sqlite3` do Python.

Esse mecanismo é preferível à cópia direta de um banco em utilização, pois permite produzir um backup consistente inclusive quando existem operações concorrentes no banco.

### 5.1. Criar a função de persistência

In [5]:
# Célula 4 — Função para salvar o banco no Google Drive

import sqlite3
import shutil
import os

from contextlib import closing


def persistir_sqlite():

    if not LOCAL_DB.exists():
        raise FileNotFoundError(
            "Banco local não encontrado."
        )

    # Arquivos temporários
    TEMP_LOCAL = (
        PROJECT_LOCAL_DIR / "backup_temp.sqlite3"
    )

    TEMP_DRIVE = (
        PROJECT_DRIVE_DIR / "banco_temp.sqlite3"
    )

    # 1. Gerar backup consistente localmente
    with closing(sqlite3.connect(LOCAL_DB)) as origem:

        with closing(
            sqlite3.connect(TEMP_LOCAL)
        ) as destino:

            origem.backup(destino)

    # 2. Verificar integridade do backup
    with closing(
        sqlite3.connect(TEMP_LOCAL)
    ) as conn:

        resultado = conn.execute(
            "PRAGMA integrity_check"
        ).fetchone()[0]

    if resultado != "ok":
        raise RuntimeError(
            f"Falha na integridade do backup: {resultado}"
        )

    # 3. Copiar backup para arquivo temporário no Drive
    shutil.copy2(
        TEMP_LOCAL,
        TEMP_DRIVE
    )

    # 4. Substituir o banco persistente
    os.replace(
        TEMP_DRIVE,
        DRIVE_DB
    )

    # 5. Remover arquivo temporário local
    TEMP_LOCAL.unlink(missing_ok=True)

    print("Banco SQLite persistido com sucesso!")
    print(f"Arquivo: {DRIVE_DB}")

O procedimento possui uma proteção importante: o arquivo persistente anterior só é substituído depois que o novo backup foi produzido, validado e copiado para o Drive.

Isso reduz o risco de perder a última versão válida durante uma operação de sincronização interrompida.

### 5.2. Criar a função de recuperação

In [6]:
## Célula 5 — Recuperar banco persistente

def recuperar_sqlite():

    if not DRIVE_DB.exists():

        print(
            "Banco persistente não encontrado."
        )

        return False

    # Abrir banco persistente e restaurar localmente
    with closing(
        sqlite3.connect(DRIVE_DB)
    ) as origem:

        with closing(
            sqlite3.connect(LOCAL_DB)
        ) as destino:

            origem.backup(destino)

    # Verificar integridade da recuperação
    with closing(
        sqlite3.connect(LOCAL_DB)
    ) as conn:

        resultado = conn.execute(
            "PRAGMA integrity_check"
        ).fetchone()[0]

    if resultado != "ok":
        raise RuntimeError(
            f"Falha na recuperação: {resultado}"
        )

    print("Banco SQLite recuperado com sucesso!")
    print(f"Arquivo local: {LOCAL_DB}")

    return True

Essa função verifica se o arquivo persistente existe.
- Se existir, recupera o banco para a instância atual.
- Se não existir, informa que ainda não há um banco persistido.

Ela não apaga nem substitui automaticamente um banco existente no Google Drive.

### 5.3. Inicialização automática do banco

Agora pode-se unir as duas funções em uma rotina de inicialização.

In [7]:
# Célula 6 — Inicializar ou recuperar o banco

if recuperar_sqlite():

    print("Projeto existente carregado.")

else:

    print("Inicializando novo banco SQLite.")

    # Criar banco vazio
    with closing(
        sqlite3.connect(LOCAL_DB)
    ) as conn:

        conn.execute("PRAGMA user_version = 1")

    # Persistir primeira versão
    persistir_sqlite()

Banco SQLite recuperado com sucesso!
Arquivo local: /content/sqlite_workspace/_base-persistencia/banco.sqlite3
Projeto existente carregado.


O comportamento esperado será:

| Situação | Comportamento |
| :--- | :--- |
| Primeira execução do projeto | Cria banco vazio e persiste no Drive
| Nova instância do Colab | Recupera o banco existente
| Reinicialização da sessão | Recupera a última versão persistida
| Novo `PROJECT_ID` | Inicializa outro banco independente

Com isso, a recuperação deixa de ser uma operação manual: basta executar as células de inicialização na ordem estabelecida.

## 6. Padronização das conexões com SQLite

A recuperação do arquivo SQLite não significa que todas as configurações de conexão serão preservadas automaticamente.

É importante distinguir:

- **Dados persistentes**: tabelas, registros, índices, views, triggers e estrutura do banco.
- **Configurações de conexão**: parâmetros aplicados quando uma conexão é aberta.

Por exemplo, a configuração `PRAGMA foreign_keys = ON` precisa ser aplicada novamente em cada conexão. Ela não é uma configuração global permanente do arquivo SQLite.

Para evitar que você precise lembrar dessas configurações em cada projeto, vamos estabelecer uma função padronizada.

In [8]:
from contextlib import contextmanager


@contextmanager
def conectar_sqlite():

    conn = sqlite3.connect(
        LOCAL_DB,
        timeout=30
    )

    # Habilitar integridade referencial
    conn.execute(
        "PRAGMA foreign_keys = ON"
    )

    # Tempo de espera para bloqueios
    conn.execute(
        "PRAGMA busy_timeout = 30000"
    )

    try:

        yield conn

        conn.commit()

    except Exception:

        conn.rollback()
        raise

    finally:

        conn.close()

A partir daqui, todas as operações SQL deverão utilizar essa função.

Exemplo:


```
with conectar_sqlite() as conn:

    conn.execute("""
        CREATE TABLE IF NOT EXISTS clientes (
            id INTEGER PRIMARY KEY,
            nome TEXT NOT NULL,
            email TEXT UNIQUE
        )
    """)
```

A função possui três responsabilidades:

1. Abrir a conexão com o banco local.
2. Aplicar as configurações necessárias.
3. Garantir o encerramento da conexão e o tratamento da transação.

O uso de `with` permite que as alterações sejam confirmadas ao final da operação ou revertidas em caso de exceção.

**Observação**: o encerramento da conexão não significa que o banco foi sincronizado com o Drive. Para isso, continua-se a utilizando `persistir_sqlite()`.

## 7. Procedimento de persistência durante os estudos

Agora que temos o banco inicializado e uma função padronizada de conexão, o fluxo operacional será bastante simples.

Sempre que realizar alterações importantes no banco, execute:

```
persistir_sqlite()
```
Exemplo de utilização:

```
with conectar_sqlite() as conn:

    conn.execute("""
        INSERT INTO clientes (nome, email)
        VALUES (?, ?)
    """, ("João", "joao@email.com"))

# Persistir alterações no Google Drive
persistir_sqlite()
```

O banco local continuará disponível para novas operações, enquanto o Drive receberá uma versão consistente e atualizada.

Quando executar a persistência?

Recomendado em três situações:

- Após criação ou alteração de estruturas
- Após inserções, atualizações ou exclusões relevantes
- Antes de encerrar ou desconectar a instância

Não é necessário sincronizar depois de cada consulta `SELECT`, pois ela não modifica os dados.

Para operações de importação de grandes conjuntos de dados, por exemplo, você pode realizar a carga completa e persistir ao final.

**Regra fundamental**: nunca considere uma alteração persistida enquanto `persistir_sqlite()` não tiver sido executada e concluída com sucesso.



## 8. Teste de persistência e recuperação

Antes de utilizar essa estrutura nos projetos reais, recomendo realizar um teste controlado.

O objetivo será simular a destruição da instância e a inicialização de uma nova.

### 8.1. Criar dados de teste

Execute:

In [9]:
with conectar_sqlite() as conn:

    conn.execute("""
        CREATE TABLE IF NOT EXISTS teste_persistencia (
            id INTEGER PRIMARY KEY,
            descricao TEXT NOT NULL
        )
    """)

    conn.execute("""
        INSERT OR REPLACE INTO teste_persistencia
        (id, descricao)
        VALUES (1, 'Persistência SQLite validada')
    """)

print("Dados de teste inseridos.")

Dados de teste inseridos.


### 8.2. Persistir o banco

In [10]:
persistir_sqlite()

Banco SQLite persistido com sucesso!
Arquivo: /content/drive/MyDrive/Colab_Notebooks/SQLite_Projetos/_base-persistencia/banco.sqlite3


Verifique se a mensagem de sucesso foi apresentada.

### 8.3. Simular destruição da instância

**Atenção**: Essa operação é exclusivamente para o teste. Não execute durante o trabalho normal, pois apagará o banco local atual.

Para simular a perda do ambiente local, remova somente o arquivo de trabalho:

In [11]:
LOCAL_DB.unlink()

print("Banco local removido.")

Banco local removido.


### 8.4. Recuperar o banco

In [12]:
recuperar_sqlite()

Banco SQLite recuperado com sucesso!
Arquivo local: /content/sqlite_workspace/_base-persistencia/banco.sqlite3


True

A função deverá reconstruir o arquivo local utilizando a versão armazenada no Google Drive.

### 8.5. Validar os dados recuperados

In [13]:
with conectar_sqlite() as conn:

    resultado = conn.execute("""
        SELECT *
        FROM teste_persistencia
    """).fetchall()

    print(resultado)

[(1, 'Persistência SQLite validada')]


Resultado esperado: `[(1, 'Persistência SQLite validada')]`

Se o resultado for apresentado corretamente, você terá validado o ciclo fundamental de persistência e recuperação.

## 9. Procedimento operacional para cada nova instância

Depois de concluir a configuração, a rotina de utilização será sempre a mesma.

Inicialização da instância:
1. Montar o Google Drive > Executar a Célula 1.
2. Definir o projeto > Executar as Células 2 e 3, mantendo o identificador correto.
3. Carregar as funções de persistência > Executar as Células 4, 5 e 7.
4. Inicializar ou recuperar o banco > Executar a Célula 6.
5. Continuar os estudos > Executar suas consultas e operações SQL utilizando `conectar_sqlite()`.

Finalização da instância:
1. Persistir o banco > Executar persistir_sqlite().
2. Encerrar o ambiente > Desconectar ou encerrar a instância do Colab normalmente.

Na próxima inicialização, o procedimento começa novamente pela montagem do Drive e recuperação do banco.

## 10. Cuidados importantes para a integridade dos dados

### 10.1. Não trabalhar simultaneamente no mesmo projeto

Evite abrir duas instâncias independentes do Colab utilizando o mesmo PROJECT_ID.

Imagine a seguinte situação:

- Instância A recupera a versão 1.
- Instância B também recupera a versão 1.
- Instância A modifica os dados e persiste a versão 2.
- Instância B modifica outros dados e persiste sua versão 2.

Nesse cenário, a persistência da instância B poderá sobrescrever as alterações realizadas pela instância A.

Nossa arquitetura utiliza cópias locais independentes, portanto não implementa sincronização de alterações entre instâncias.

**Recomendação**: uma instância ativa por projeto.

### 10.2. Não excluir arquivos auxiliares do SQLite manualmente

O SQLite pode utilizar arquivos auxiliares durante sua operação, dependendo do modo de journal e das transações.

Por isso:

- Não copie manualmente um banco enquanto estiver sendo utilizado.
- Não exclua arquivos `-wal` ou `-shm` manualmente.
- Utilize `persistir_sqlite()` para gerar backups.
- Encerre as conexões antes de realizar a sincronização.

A implementação explicitada anteriormente utiliza o mecanismo de backup nativo do SQLite justamente para evitar depender da cópia direta do arquivo em utilização.

### 10.3. Manter cópias históricas dos bancos

A implementação apresentada mantém uma versão principal do banco por projeto.

Para estudos mais importantes, é recomendado acrescentar uma rotina de backups históricos, por exemplo:

```
projeto_01/
│
├── banco.sqlite3
│
└── backups/
    ├── backup_20261028_130000.sqlite3
    ├── backup_20261029_150000.sqlite3
    └── backup_20261030_180000.sqlite3
```

Isso permitirá recuperar versões anteriores caso ocorra uma exclusão acidental, alteração incorreta ou corrupção lógica dos dados.

Essa funcionalidade não é obrigatória para a configuração inicial, mas será um aprimoramento estratégico.

### 10.4. Entender os limites da persistência

O Colab executa seus códigos em máquinas virtuais que podem ser excluídas após períodos de inatividade ou pelo limite de vida da instância. O Google Drive, por sua vez, possui particularidades de desempenho e operações de entrada e saída.

Portanto, a persistência depende de duas condições:

1. O Google Drive estar corretamente montado e acessível.
2. A última operação de backup ter sido concluída com sucesso.

Não existe garantia de recuperação de alterações que permaneceram exclusivamente na máquina virtual e não foram sincronizadas.

# Conectando o Google Colab ao GitHub

## Criar repositório no GitHub

Depois de criar sua conta no GitHub, o primeiro passo para compartilhar seu notebook é criar um repositório (https://github.com/new) .

O repositório será o espaço no seu GitHub onde seu projeto será armazenado. Geralmente, é criado um novo repositório para cada projeto diferente.

Nesta tela você deverá nomear o repositório que deseja criar. Você pode escolher um nome que considere mais apropriado. Importante ressaltar que, ao nomear o repositório, evite espaços ou caracteres especiais, pois esse nome fará parte da URL para acessar seu repositório.

Depois de nomear o repositório, é crucial lembrar de marcar a caixa de seleção descrita como “Add a README file”. Com essa opção marcada, ao criar seu repositório, automaticamente será gerado um arquivo no formato MD (Markdown). Este arquivo `README.md` será usado posteriormente para fornecer uma descrição do projeto que está armazenado naquele repositório.

## Salvar o notebook do Google Colab no GitHub


O Colab possui uma integração direta com o GitHub que dispensa o uso de comandos de terminal para operações básicas.

- Para salvar um notebook no GitHub:

	1. No menu superior, clique em Arquivo > Salvar uma cópia no GitHub.
	2. Autorize o Colab a acessar sua conta do GitHub (se for a primeira vez).
	3. Escolha o Repositório e a Branch.
	4. Ative a opção "Incluir link para o Colab" (opcional, adiciona um botão no topo do arquivo que abre o notebook direto no Colab).
	5. Clique em OK.

- Para abrir um notebook do GitHub:

  1. Vá em Arquivo > Abrir notebook.
	2. Selecione a aba GitHub.
	3. Cole a URL do repositório ou pesquise pelo nome do usuário/organização.

Como funciona o versionamento: Cada vez que você clica em "Salvar uma cópia no GitHub", o Colab cria um novo commit diretamente na branch selecionada. Se você salvar o mesmo arquivo várias vezes, o histórico de commits do GitHub registrará cada alteração normalmente.